In [ ]:
# In[1]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Load files
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to compute compact summary for a dataframe
def compact_summary(df, name_field=None, message_field=None):
    # counts and timestamps
    row_count = len(df)
    if 'timestamp' in df.columns and row_count > 0:
        min_ts = pd.to_datetime(df['timestamp'].min(), unit='s', utc=True).tz_convert(tz)
        max_ts = pd.to_datetime(df['timestamp'].max(), unit='s', utc=True).tz_convert(tz)
    else:
        min_ts = pd.NaT
        max_ts = pd.NaT

    # unique cmdb ids (first 50)
    if 'cmdb_id' in df.columns:
        uniq_cmdb = pd.unique(df['cmdb_id']).tolist()[:50]
        # represent as a short pandas Series for compact display
        uniq_cmdb_s = pd.Series(uniq_cmdb, name='unique_cmdb_id_first50')
    else:
        uniq_cmdb_s = pd.Series([], name='unique_cmdb_id_first50', dtype=object)

    # top name counts depending on available field
    if name_field and name_field in df.columns:
        top_names = df[name_field].fillna('<NA>').astype(str).value_counts().head(50)
        top_names_df = top_names.rename_axis(name_field).reset_index(name='count')
    elif message_field and message_field in df.columns:
        # For error messages: show truncated message preview to keep compact
        msgs = df[message_field].fillna('').astype(str)
        # create truncated preview column for grouping to keep compact
        preview = msgs.str.slice(0, 80)
        top_msgs = preview.value_counts().head(50)
        top_names_df = top_msgs.rename_axis('message_preview_80').reset_index(name='count')
    else:
        top_names_df = pd.DataFrame(columns=['name', 'count'])

    # examples (first 5 rows)
    examples = df.head(5)

    # summary dataframe (single-row) with compact fields
    summary_df = pd.DataFrame({
        'row_count': [row_count],
        'min_timestamp': [min_ts],
        'max_timestamp': [max_ts],
        'unique_cmdb_count': [len(pd.unique(df['cmdb_id'])) if 'cmdb_id' in df.columns else 0],
    })

    return summary_df, uniq_cmdb_s, top_names_df, examples

# Metric file summary (kpi_name)
metric_summary, metric_cmdbs, metric_top_kpis, metric_examples = compact_summary(df_metric, name_field='kpi_name')

# Trace file summary (trace_name)
trace_summary, trace_cmdbs, trace_top_traces, trace_examples = compact_summary(df_trace, name_field='trace_name')

# Log file summary (log_name)
log_summary, log_cmdbs, log_top_logs, log_examples = compact_summary(df_log, name_field='log_name')

# Error logs summary (use message preview)
error_summary, error_cmdbs, error_top_msgs, error_examples = compact_summary(df_error, message_field='message')

# Display compact outputs (multiple variables separated by commas per IPython convention)
metric_summary, metric_cmdbs, metric_top_kpis.head(50), metric_examples, \
trace_summary, trace_cmdbs, trace_top_traces.head(50), trace_examples, \
log_summary, log_cmdbs, log_top_logs.head(50), log_examples, \
error_summary, error_cmdbs, error_top_msgs.head(50), error_examples

```
Out[1]:
```
```python
# Summarize execution results in plain English using existing variables from the kernel
summary = (
    "metric.csv: total rows=28812; time range=2021-03-04 22:00:00+08:00 to 2021-03-04 22:30:00+08:00; "
    "unique cmdb_ids=29 (examples: IG01, IG02, MG01, MG02, Mysql01, ...); "
    "top 5 kpi_names by frequency: "
    "1) OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT (375), "
    "2) OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT (374), "
    "3) OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies (373), "
    "4) OSLinux-OSLinux_ZABBIX_Host_Uptime (373), "
    "5) OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount (373). "
    "Example rows include JVM-Memory and JVM-Runtime KPIs."
    "\n\n"
    "trace.csv: total rows=6492; time range=2021-03-04 22:00:00+08:00 to 2021-03-04 22:29:00+08:00; "
    "unique cmdb_ids=12 (examples: IG01, IG02, MG01, MG02, Tomcat01, ...); "
    "top 5 trace_names by frequency: "
    "1) trace.self.duration_mean (360), "
    "2) trace.self.duration_p95 (360), "
    "3) trace.self.row_count (360), "
    "4) trace.to_MG01.row_count (218), "
    "5) trace.to_MG01.duration_mean (218). "
    "Example rows include trace.self.duration_mean and trace.to_Tomcat01 entries."
    "\n\n"
    "log.csv: total rows=0; no timestamps or cmdb_ids present; no log_name entries; no examples."
    "\n\n"
    "error_logs.csv: total rows=0; no timestamps or cmdb_ids present; no messages; no examples."
)

# Display concise summary
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   row_count             min_timestamp             max_timestamp  unique_cmdb_count
0      28812 2021-03-04 22:00:00+08:00 2021-03-04 22:30:00+08:00                 29, 0         IG01
1         IG02
2         MG01
3         MG02
4      Mysql01
        ...   
24    apache02
25    dockerA1
26    dockerA2
27    dockerB1
28    dockerB2
Name: unique_cmdb_id_first50, Length: 29, dtype: object,                                              kpi_name  count
0      OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT    375
1        OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT    374
2       OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies    373
3                  OSLinux-OSLinux_ZABBIX_Host_Uptime    373
4          OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount    373
..                                                ...    ...
45         OSLinux-OSLinux_FILESYSTEM_-tmp_FSCapacity    185
46                    OSLinux-CPU_CPU-2_SingleCpuidle    185
47  OSLinux-OSLinux_FILESYSTEM_-cmbc_admin_FSInode...    185
48     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    184
49  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKPer...    184

[50 rows x 2 columns],     timestamp cmdb_id                                     kpi_name         value
0  1614866400    IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09
1  1614866400    IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  7.167304e+08
2  1614866400    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  8.265914e+08
3  1614866400    IG01              JVM-Runtime_7778_JVM_JVM_Uptime  1.929684e+06
4  1614866400    IG01     JVM-Threads_7778_JVM_ThreadCount_Threads  8.700000e+01,    row_count             min_timestamp             max_timestamp  unique_cmdb_count
0       6492 2021-03-04 22:00:00+08:00 2021-03-04 22:29:00+08:00                 12, 0         IG01
1         IG02
2         MG01
3         MG02
4     Tomcat01
5     Tomcat02
6     Tomcat03
7     Tomcat04
8     dockerA1
9     dockerA2
10    dockerB1
11    dockerB2
Name: unique_cmdb_id_first50, dtype: object,                          trace_name  count
0          trace.self.duration_mean    360
1           trace.self.duration_p95    360
2              trace.self.row_count    360
3           trace.to_MG01.row_count    218
4       trace.to_MG01.duration_mean    218
..                              ...    ...
45    trace.from_root.duration_mean     54
46        trace.from_root.row_count     54
47     trace.from_root.duration_p95     54
48   trace.to_dockerA1.duration_p95     50
49  trace.to_dockerA1.duration_mean     50

[50 rows x 2 columns],     timestamp cmdb_id                       trace_name       value
0  1614866400    IG01         trace.self.duration_mean    0.090859
1  1614866400    IG01          trace.self.duration_p95    0.455000
2  1614866400    IG01             trace.self.row_count  298.000000
3  1614866400    IG01  trace.to_Tomcat01.duration_mean    0.084250
4  1614866400    IG01   trace.to_Tomcat01.duration_p95    0.360800,    row_count min_timestamp max_timestamp  unique_cmdb_count
0          0           NaT           NaT                  0, Series([], Name: unique_cmdb_id_first50, dtype: object), Empty DataFrame
Columns: [log_name, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, log_name, value]
Index: [],    row_count min_timestamp max_timestamp  unique_cmdb_count
0          0           NaT           NaT                  0, Series([], Name: unique_cmdb_id_first50, dtype: object), Empty DataFrame
Columns: [message_preview_80, count]
Index: [], Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing dataframes: df_metric, df_trace
# Ensure 'value' is numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Time window (inclusive)
start_ts = 1614866400
end_ts = 1614868200

def analyze_series(df, key_cols, name_col):
    # Compute global P95 per group using entire series (rule 9)
    grp = df.dropna(subset=['value']).groupby(key_cols)['value']
    thresholds = grp.quantile(0.95).reset_index().rename(columns={'value': 'global_P95'})
    
    # Merge thresholds back
    df_thr = df.merge(thresholds, on=key_cols, how='left')
    
    # Filter to time window
    window = df_thr[(df_thr['timestamp'] >= start_ts) & (df_thr['timestamp'] <= end_ts)].copy()
    
    # For each group compute required aggregates using P95
    def agg_using_threshold(dfw, thr_col):
        # anomaly condition value >= global_P95
        cond = dfw['value'] >= dfw[thr_col]
        # group
        g = dfw.groupby(key_cols)
        res = g.apply(lambda gdf: pd.Series({
            'global_P95': gdf[thr_col].iloc[0],
            'anomaly_count_in_window': int((gdf['value'] >= gdf[thr_col].iloc[0]).sum()),
            'earliest_anomaly_timestamp': int(gdf.loc[gdf['value'] >= gdf[thr_col].iloc[0], 'timestamp'].min()) if (gdf['value'] >= gdf[thr_col].iloc[0]).any() else pd.NA,
            'max_value_in_window': gdf['value'].max(),
            'total_points_in_window': int(len(gdf))
        })).reset_index()
        return res

    if window.empty:
        result_p95 = pd.DataFrame(columns=key_cols + ['global_P95','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window','total_points_in_window'])
    else:
        result_p95 = agg_using_threshold(window, 'global_P95')
    
    # Determine if any anomalies found
    total_anomalies = result_p95['anomaly_count_in_window'].sum() if not result_p95.empty else 0
    used_quantile = 'P95'
    
    # If no anomalies at P95, fallback to P90 (per instruction)
    if total_anomalies == 0:
        # recompute thresholds at P90 using entire series (rule 9)
        thresholds_p90 = df.dropna(subset=['value']).groupby(key_cols)['value'].quantile(0.90).reset_index().rename(columns={'value':'global_P90'})
        df_thr_p90 = df.merge(thresholds_p90, on=key_cols, how='left')
        window_p90 = df_thr_p90[(df_thr_p90['timestamp'] >= start_ts) & (df_thr_p90['timestamp'] <= end_ts)].copy()
        if window_p90.empty:
            result = pd.DataFrame(columns=key_cols + ['global_P90','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window','total_points_in_window'])
        else:
            # aggregate using P90
            def agg_p90(dfw):
                g = dfw.groupby(key_cols)
                res = g.apply(lambda gdf: pd.Series({
                    'global_P90': gdf['global_P90'].iloc[0],
                    'anomaly_count_in_window': int((gdf['value'] >= gdf['global_P90'].iloc[0]).sum()),
                    'earliest_anomaly_timestamp': int(gdf.loc[gdf['value'] >= gdf['global_P90'].iloc[0], 'timestamp'].min()) if (gdf['value'] >= gdf['global_P90'].iloc[0]).any() else pd.NA,
                    'max_value_in_window': gdf['value'].max(),
                    'total_points_in_window': int(len(gdf))
                })).reset_index()
                return res
            result = agg_p90(window_p90)
        used_quantile = 'P90'
    else:
        result = result_p95.rename(columns={'global_P95':'global_P95'})
    
    # Normalize column names to required output and convert earliest timestamp to timezone-aware datetime strings
    if not result.empty:
        # Ensure the quantile column name is consistent
        if 'global_P95' in result.columns:
            result = result.rename(columns={'global_P95':'global_threshold'})
        elif 'global_P90' in result.columns:
            result = result.rename(columns={'global_P90':'global_threshold'})
        # convert earliest_anomaly_timestamp to tz-aware datetimes or keep NA
        def to_dt(x):
            if pd.isna(x):
                return pd.NaT
            return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)
        result['earliest_anomaly_timestamp'] = result['earliest_anomaly_timestamp'].apply(lambda x: to_dt(x))
        # reorder columns as requested
        out_cols = key_cols + ['global_threshold','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window','total_points_in_window']
        result = result[out_cols]
        # sort by anomaly_count desc, then by max_value desc
        result = result.sort_values(['anomaly_count_in_window','max_value_in_window'], ascending=[False, False]).reset_index(drop=True)
    else:
        # create empty frame with proper columns
        out_cols = key_cols + ['global_threshold','anomaly_count_in_window','earliest_anomaly_timestamp','max_value_in_window','total_points_in_window']
        result = pd.DataFrame(columns=out_cols)
    
    return result, used_quantile

# Metric analysis
metric_result, metric_used = analyze_series(df_metric, ['cmdb_id','kpi_name'], 'kpi_name')

# Trace analysis
trace_result, trace_used = analyze_series(df_trace, ['cmdb_id','trace_name'], 'trace_name')

# Keep outputs compact: top 20 rows each and notes indicating P95 or P90 used
metric_note = f"Metric analysis used {metric_used} (global thresholds computed from full series)."
trace_note = f"Trace analysis used {trace_used} (global thresholds computed from full series)."

# Final displayed variables (compact)
metric_note, metric_result.head(20), trace_note, trace_result.head(20)

```
Out[2]:
```
```python
# Summarize the analysis results in plain English using existing kernel variables
metric_top = metric_result.head(5)
trace_top = trace_result.head(5)

lines = []
lines.append(f"Metric analysis used {metric_used}. Top metric anomaly pairs (top 5):")

for i, row in metric_top.reset_index(drop=True).iterrows():
    lines.append(
        f"{i+1}) {row['cmdb_id']} | {row['kpi_name']} | global_threshold={row['global_threshold']:.6g} | "
        f"anomalies={int(row['anomaly_count_in_window'])} | earliest={row['earliest_anomaly_timestamp']} | "
        f"max={row['max_value_in_window']:.6g} | points={int(row['total_points_in_window'])}"
    )

lines.append("") 
lines.append(f"Trace analysis used {trace_used}. Top trace anomaly pairs (top 5):")

for i, row in trace_top.reset_index(drop=True).iterrows():
    lines.append(
        f"{i+1}) {row['cmdb_id']} | {row['trace_name']} | global_threshold={row['global_threshold']:.6g} | "
        f"anomalies={int(row['anomaly_count_in_window'])} | earliest={row['earliest_anomaly_timestamp']} | "
        f"max={row['max_value_in_window']:.6g} | points={int(row['total_points_in_window'])}"
    )

lines.append("")
lines.append("Note: Global thresholds were computed from the entire series before window filtering. "
             "P95 thresholds were used (no fallback to P90 was necessary).")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

('Metric analysis used P95 (global thresholds computed from full series).',      cmdb_id                                           kpi_name  global_threshold  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window  total_points_in_window
0    Redis01                redis-Redis_6379_Redis  (maxmemory)      4.294967e+09                     31.0  2021-03-04 22:00:00+08:00         4.294967e+09                    31.0
1    Redis02                redis-Redis_6379_Redis  (maxmemory)      4.294967e+09                     31.0  2021-03-04 22:00:00+08:00         4.294967e+09                    31.0
2       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax      2.040136e+09                     31.0  2021-03-04 22:00:00+08:00         2.040136e+09                    31.0
3       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax      2.040136e+09                     31.0  2021-03-04 22:00:00+08:00         2.040136e+09                    31.0
4       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax      2.040136e+09                     31.0  2021-03-04 22:00:00+08:00         2.040136e+09                    31.0
5       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed      9.468300e+08                     31.0  2021-03-04 22:00:00+08:00         9.468300e+08                    31.0
6       IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed      8.239351e+08                     31.0  2021-03-04 22:00:00+08:00         8.239351e+08                    31.0
7    Redis02         redis-Redis_6379_Redis  (used_memory_peak)      9.570678e+07                     31.0  2021-03-04 22:00:00+08:00         9.570678e+07                    31.0
8    Redis01         redis-Redis_6379_Redis  (used_memory_peak)      3.487778e+07                     31.0  2021-03-04 22:00:00+08:00         3.487778e+07                    31.0
9   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...      5.781400e+04                     31.0  2021-03-04 22:00:00+08:00         5.781400e+04                    31.0
10  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...      5.651600e+04                     31.0  2021-03-04 22:00:00+08:00         5.651600e+04                    31.0
11  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...      5.560700e+04                     31.0  2021-03-04 22:00:00+08:00         5.560700e+04                    31.0
12  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...      5.143800e+04                     31.0  2021-03-04 22:00:00+08:00         5.143800e+04                    31.0
13   Redis01          redis-Redis_6379_Redis  (keyspace_misses)      6.938000e+03                     31.0  2021-03-04 22:00:00+08:00         6.938000e+03                    31.0
14   Redis01        redis-Redis_6379_Redis  (connected_clients)      6.020000e+02                     31.0  2021-03-04 22:00:00+08:00         6.020000e+02                    31.0
15  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...      5.000000e+02                     31.0  2021-03-04 22:00:00+08:00         5.000000e+02                    31.0
16  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...      5.000000e+02                     31.0  2021-03-04 22:00:00+08:00         5.000000e+02                    31.0
17  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...      5.000000e+02                     31.0  2021-03-04 22:00:00+08:00         5.000000e+02                    31.0
18  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...      5.000000e+02                     31.0  2021-03-04 22:00:00+08:00         5.000000e+02                    31.0
19   Redis02        redis-Redis_6379_Redis  (connected_clients)      4.020000e+02                     31.0  2021-03-04 22:00:00+08:00         4.020000e+02                    31.0, 'Trace analysis used P95 (global thresholds computed from full series).',      cmdb_id                     trace_name  global_threshold  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window  total_points_in_window
0   dockerB2        trace.self.duration_p95             0.012                     10.0  2021-03-04 22:01:00+08:00                0.013                    30.0
1   dockerB1        trace.self.duration_p95             0.014                      7.0  2021-03-04 22:00:00+08:00                0.014                    30.0
2       MG01    trace.to_dockerA1.row_count            28.000                      5.0  2021-03-04 22:03:00+08:00               29.000                    25.0
3   dockerA1      trace.from_MG01.row_count            28.000                      5.0  2021-03-04 22:03:00+08:00               29.000                    25.0
4       IG01    trace.to_Tomcat04.row_count            41.000                      4.0  2021-03-04 22:00:00+08:00               42.000                    30.0
5   Tomcat04      trace.from_IG01.row_count            41.000                      4.0  2021-03-04 22:00:00+08:00               42.000                    30.0
6       MG02  trace.from_dockerB1.row_count            43.000                      3.0  2021-03-04 22:06:00+08:00               50.000                    25.0
7   dockerB1        trace.to_MG02.row_count            43.000                      3.0  2021-03-04 22:06:00+08:00               50.000                    25.0
8       IG01    trace.to_Tomcat03.row_count            45.000                      3.0  2021-03-04 22:03:00+08:00               49.000                    30.0
9   Tomcat03      trace.from_IG01.row_count            45.000                      3.0  2021-03-04 22:03:00+08:00               49.000                    30.0
10      IG02    trace.to_Tomcat01.row_count            40.000                      3.0  2021-03-04 22:00:00+08:00               45.000                    30.0
11  Tomcat01      trace.from_IG02.row_count            40.000                      3.0  2021-03-04 22:00:00+08:00               45.000                    30.0
12      IG02    trace.to_Tomcat03.row_count            43.000                      3.0  2021-03-04 22:00:00+08:00               44.000                    30.0
13  Tomcat03      trace.from_IG02.row_count            43.000                      3.0  2021-03-04 22:00:00+08:00               44.000                    30.0
14      IG01    trace.to_Tomcat01.row_count            42.000                      3.0  2021-03-04 22:03:00+08:00               42.000                    30.0
15  Tomcat01      trace.from_IG01.row_count            42.000                      3.0  2021-03-04 22:03:00+08:00               42.000                    30.0
16      MG01  trace.from_dockerB2.row_count            39.000                      3.0  2021-03-04 22:06:00+08:00               39.000                    25.0
17  dockerB2        trace.to_MG01.row_count            39.000                      3.0  2021-03-04 22:06:00+08:00               39.000                    25.0
18      MG01    trace.to_dockerB2.row_count            37.000                      3.0  2021-03-04 22:13:00+08:00               38.000                    25.0
19  dockerB2      trace.from_MG01.row_count            37.000                      3.0  2021-03-04 22:13:00+08:00               38.000                    25.0)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse timezone and window bounds from previous kernel state
# tz, start_ts, end_ts, df_metric, df_trace, metric_result, trace_result are available

# Helper to compute severity and longest consecutive anomaly run per group
def augment_with_sequence_info(df_full, agg_df, key_cols, name_col):
    # Merge to get rows for groups present in agg_df and attach global_threshold
    merged = df_full.merge(agg_df[key_cols + ['global_threshold']], on=key_cols, how='inner')
    # Filter to the window (ensure using same bounds)
    window_rows = merged[(merged['timestamp'] >= start_ts) & (merged['timestamp'] <= end_ts)].copy()
    # Ensure numeric values
    window_rows['value'] = pd.to_numeric(window_rows['value'], errors='coerce')
    # Sort rows per group
    window_rows = window_rows.sort_values(key_cols + ['timestamp'])
    
    results = []

    # Group iteration (only groups present in agg_df)
    for keys, g in window_rows.groupby(key_cols):
        g = g.sort_values('timestamp')
        global_thr = float(g['global_threshold'].iloc[0])
        max_val = float(g['value'].max()) if not g['value'].empty else np.nan
        total_points = int(len(g))
        # Anomaly boolean series
        anomaly = (g['value'] >= global_thr).astype(int)
        # Compute longest consecutive run where consecutive means timestamp diff == 60 seconds
        prev_ts = None
        cur_run = 0
        cur_run_start = None
        best_run = 0
        best_run_start_ts = None

        # iterate rows
        for idx, row in g[['timestamp','value']].iterrows():
            ts = int(row['timestamp'])
            is_anom = int(row['value'] >= global_thr)
            if is_anom:
                if prev_ts is None or (ts - prev_ts == 60):
                    # continue run
                    if cur_run == 0:
                        cur_run_start = ts
                    cur_run += 1
                else:
                    # gap -> new run
                    cur_run = 1
                    cur_run_start = ts
            else:
                cur_run = 0
                cur_run_start = None
            # update best
            if cur_run > best_run:
                best_run = cur_run
                best_run_start_ts = cur_run_start
            prev_ts = ts

        # earliest timestamp of longest run as tz-aware datetime or NaT
        if best_run_start_ts is None:
            earliest_run_dt = pd.NaT
        else:
            earliest_run_dt = pd.to_datetime(int(best_run_start_ts), unit='s', utc=True).tz_convert(tz)
        # anomaly count in window (recompute)
        anomaly_count = int((g['value'] >= global_thr).sum())
        # severity_ratio handling (null if global_threshold == 0)
        if global_thr == 0:
            severity = np.nan
        else:
            severity = (max_val - global_thr) / global_thr

        # Build result record
        record = dict(zip(key_cols, keys if isinstance(keys, tuple) else (keys,)))
        record.update({
            'global_threshold': global_thr,
            'max_value_in_window': max_val,
            'severity_ratio': severity,
            'longest_consecutive_anomaly_length': int(best_run),
            'earliest_timestamp_of_longest_run': earliest_run_dt,
            'anomaly_count_in_window': anomaly_count,
            'total_points_in_window': total_points
        })
        results.append(record)

    # Build DataFrame
    if results:
        out = pd.DataFrame(results)
    else:
        # empty with columns
        out_cols = key_cols + ['global_threshold','max_value_in_window','severity_ratio',
                               'longest_consecutive_anomaly_length','earliest_timestamp_of_longest_run',
                               'anomaly_count_in_window','total_points_in_window']
        out = pd.DataFrame(columns=out_cols)
    # Some groups in agg_df might have zero total_points_in_window (unlikely here) - ensure they are included
    # Merge to include any missing groups from agg_df (keeping their existing anomaly counts etc.)
    out = agg_df.merge(out, on=key_cols, how='left', suffixes=('_orig',''))
    # If merge introduced duplicate columns, prefer computed ones; fill missing from agg_df
    # Keep final column set as requested; rename name column appropriately
    # Requested columns depend on name_col present in key_cols
    final_cols = key_cols + [
        'global_threshold','max_value_in_window','severity_ratio',
        'longest_consecutive_anomaly_length','earliest_timestamp_of_longest_run',
        'anomaly_count_in_window','total_points_in_window'
    ]
    # Ensure columns exist
    for c in final_cols:
        if c not in out.columns:
            out[c] = pd.NA

    # Convert earliest_timestamp_of_longest_run to datetime tz-aware (already done), keep as is
    # Sort by longest_consecutive_anomaly_length desc then severity_ratio desc
    out_sorted = out[final_cols].sort_values(['longest_consecutive_anomaly_length','severity_ratio'],
                                            ascending=[False, False]).reset_index(drop=True)
    return out_sorted

# Compute augmented tables for metrics and traces
metric_augmented = augment_with_sequence_info(df_metric, metric_result, ['cmdb_id','kpi_name'], 'kpi_name')
trace_augmented = augment_with_sequence_info(df_trace, trace_result, ['cmdb_id','trace_name'], 'trace_name')

# Limit to top 20 rows each (already sorted)
metric_aug_top20 = metric_augmented.head(20)
trace_aug_top20 = trace_augmented.head(20)

# Display compact outputs (variable names)
metric_aug_top20, trace_aug_top20

```
Out[3]:
```
```python
# Build a concise plain-English summary string from existing kernel tables
metric_top5 = metric_aug_top20.head(5)
trace_top5 = trace_aug_top20.head(5)

lines = []
lines.append("Summary (analysis used P95 global thresholds; window = 2021-03-04 22:00:00 to 22:30:00 +08:00):")
lines.append("")
lines.append("Metrics — top 5 anomaly pairs:")
for i, r in metric_top5.reset_index(drop=True).iterrows():
    lines.append(
        f"{i+1}) {r['cmdb_id']} | {r['kpi_name']} | threshold={r['global_threshold']:.6g} | "
        f"max={r['max_value_in_window']:.6g} | severity_ratio={r['severity_ratio'] if pd.notna(r['severity_ratio']) else 'null'} | "
        f"longest_consecutive={int(r['longest_consecutive_anomaly_length'])} (start={r['earliest_timestamp_of_longest_run']}) | "
        f"anomalies={int(r['anomaly_count_in_window'])} / points={int(r['total_points_in_window'])}"
    )

lines.append("")
lines.append("Traces — top 5 anomaly pairs:")
for i, r in trace_top5.reset_index(drop=True).iterrows():
    lines.append(
        f"{i+1}) {r['cmdb_id']} | {r['trace_name']} | threshold={r['global_threshold']:.6g} | "
        f"max={r['max_value_in_window']:.6g} | severity_ratio={r['severity_ratio'] if pd.notna(r['severity_ratio']) else 'null'} | "
        f"longest_consecutive={int(r['longest_consecutive_anomaly_length'])} (start={r['earliest_timestamp_of_longest_run']}) | "
        f"anomalies={int(r['anomaly_count_in_window'])} / points={int(r['total_points_in_window'])}"
    )

lines.append("")
lines.append("Key takeaways:")
lines.append("- Many metric pairs (e.g., Redis memory and several JVM/Tomcat KPIs) show full-window consecutive anomalies of length 31 minutes,")
lines.append("  but severity_ratio = 0.0 for those rows (observed max equals the global P95 threshold).")
lines.append("- Trace anomalies are shorter and more variable (longest consecutive run observed = 5 minutes for dockerB2.trace.self.duration_p95),")
lines.append("  and show non-zero severity ratios (meaning observed peaks exceed the P95 threshold).")
lines.append("- All global thresholds were computed from the full series before window filtering; no P90 fallback was needed.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  global_threshold  max_value_in_window  severity_ratio  longest_consecutive_anomaly_length earliest_timestamp_of_longest_run  anomaly_count_in_window  total_points_in_window
0    Redis01                redis-Redis_6379_Redis  (maxmemory)      4.294967e+09         4.294967e+09             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
1    Redis02                redis-Redis_6379_Redis  (maxmemory)      4.294967e+09         4.294967e+09             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
2       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax      2.040136e+09         2.040136e+09             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
3       IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax      2.040136e+09         2.040136e+09             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
4       MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax      2.040136e+09         2.040136e+09             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
5       MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed      9.468300e+08         9.468300e+08             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
6       IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed      8.239351e+08         8.239351e+08             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
7    Redis02         redis-Redis_6379_Redis  (used_memory_peak)      9.570678e+07         9.570678e+07             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
8    Redis01         redis-Redis_6379_Redis  (used_memory_peak)      3.487778e+07         3.487778e+07             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
9   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...      5.781400e+04         5.781400e+04             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
10  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...      5.651600e+04         5.651600e+04             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
11  Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...      5.560700e+04         5.560700e+04             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
12  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...      5.143800e+04         5.143800e+04             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
13   Redis01          redis-Redis_6379_Redis  (keyspace_misses)      6.938000e+03         6.938000e+03             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
14   Redis01        redis-Redis_6379_Redis  (connected_clients)      6.020000e+02         6.020000e+02             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
15  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...      5.000000e+02         5.000000e+02             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
16  Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...      5.000000e+02         5.000000e+02             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
17  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...      5.000000e+02         5.000000e+02             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
18  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...      5.000000e+02         5.000000e+02             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31
19   Redis02        redis-Redis_6379_Redis  (connected_clients)      4.020000e+02         4.020000e+02             0.0                                  31         2021-03-04 22:00:00+08:00                       31                      31,      cmdb_id                         trace_name  global_threshold  max_value_in_window  severity_ratio  longest_consecutive_anomaly_length earliest_timestamp_of_longest_run  anomaly_count_in_window  total_points_in_window
0   dockerB2            trace.self.duration_p95          0.012000             0.013000        0.083333                                   5         2021-03-04 22:01:00+08:00                       10                      30
1   dockerB1            trace.self.duration_p95          0.014000             0.014000        0.000000                                   3         2021-03-04 22:25:00+08:00                        7                      30
2       MG02   trace.from_Tomcat01.duration_p95          0.389955             0.943000        1.418228                                   2         2021-03-04 22:06:00+08:00                        2                      30
3   Tomcat01         trace.to_MG02.duration_p95          0.389955             0.943000        1.418228                                   2         2021-03-04 22:06:00+08:00                        2                      30
4       MG01            trace.self.duration_p95          0.504970             1.043250        1.065964                                   2         2021-03-04 22:07:00+08:00                        2                      30
5       MG01   trace.from_dockerA1.duration_p95          0.234105             0.468650        1.001879                                   2         2021-03-04 22:19:00+08:00                        2                      24
6   dockerA1         trace.to_MG01.duration_p95          0.234105             0.468650        1.001879                                   2         2021-03-04 22:19:00+08:00                        2                      24
7       MG01  trace.from_dockerB2.duration_mean          0.159721             0.316333        0.980535                                   2         2021-03-04 22:07:00+08:00                        2                      25
8   dockerB2        trace.to_MG01.duration_mean          0.159721             0.316333        0.980535                                   2         2021-03-04 22:07:00+08:00                        2                      25
9       MG02  trace.from_Tomcat04.duration_mean          0.120005             0.231139        0.926082                                   2         2021-03-04 22:09:00+08:00                        2                      30
10  Tomcat04        trace.to_MG02.duration_mean          0.120005             0.231139        0.926082                                   2         2021-03-04 22:09:00+08:00                        2                      30
11      IG01     trace.to_Tomcat02.duration_p95          0.947960             1.667600        0.759146                                   2         2021-03-04 22:07:00+08:00                        2                      30
12  Tomcat02       trace.from_IG01.duration_p95          0.947960             1.667600        0.759146                                   2         2021-03-04 22:07:00+08:00                        2                      30
13      MG01   trace.from_dockerB2.duration_p95          0.597800             1.037600        0.735698                                   2         2021-03-04 22:08:00+08:00                        2                      25
14  dockerB2         trace.to_MG01.duration_p95          0.597800             1.037600        0.735698                                   2         2021-03-04 22:08:00+08:00                        2                      25
15      IG01     trace.to_Tomcat04.duration_p95          0.869962             1.459950        0.678176                                   2         2021-03-04 22:08:00+08:00                        2                      30
16  Tomcat04       trace.from_IG01.duration_p95          0.869962             1.459950        0.678176                                   2         2021-03-04 22:08:00+08:00                        2                      30
17  Tomcat01           trace.self.duration_mean          0.014320             0.023657        0.652037                                   2         2021-03-04 22:06:00+08:00                        2                      30
18      MG01  trace.from_Tomcat01.duration_mean          0.093623             0.154400        0.649170                                   2         2021-03-04 22:06:00+08:00                        2                      30
19  Tomcat01        trace.to_MG01.duration_mean          0.093623             0.154400        0.649170                                   2         2021-03-04 22:06:00+08:00                        2                      30)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse kernel variables: df_trace, trace_augmented, trace_result, start_ts, end_ts, tz
# Ensure numeric
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Select top 10 candidates by severity_ratio (non-zero, not NaN)
cands = trace_augmented[trace_augmented['severity_ratio'].notna() & (trace_augmented['severity_ratio'] > 0)]
cands = cands.sort_values('severity_ratio', ascending=False).head(10).reset_index(drop=True)

records = []
for _, row in cands.iterrows():
    cmdb = row['cmdb_id']
    tname = row['trace_name']
    thr = float(row['global_threshold'])
    # Filter trace rows in the window for this pair
    subset = df_trace[
        (df_trace['cmdb_id'] == cmdb) &
        (df_trace['trace_name'] == tname) &
        (df_trace['timestamp'] >= start_ts) &
        (df_trace['timestamp'] <= end_ts)
    ].sort_values('timestamp')
    # anomaly timestamps where value >= threshold
    anom_ts = subset.loc[subset['value'] >= thr, 'timestamp'].astype('Int64').tolist()
    # limit to first 20
    anom_ts_limited = anom_ts[:20]
    # convert to tz-aware datetimes
    anom_dt = [pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz) for ts in anom_ts_limited]
    # earliest anomaly timestamp: prefer trace_result's earliest_anomaly_timestamp if present
    mask_res = (trace_result['cmdb_id'] == cmdb) & (trace_result['trace_name'] == tname)
    if mask_res.any():
        earliest = trace_result.loc[mask_res, 'earliest_anomaly_timestamp'].iloc[0]
    else:
        earliest = pd.NaT
        if anom_dt:
            earliest = anom_dt[0]
    note = "" if anom_dt else "no anomaly timestamps in window"
    records.append({
        'cmdb_id': cmdb,
        'trace_name': tname,
        'global_threshold': thr,
        'anomaly_timestamps_in_window': anom_dt,
        'earliest_anomaly_timestamp': earliest,
        'anomaly_count_in_window': int(row['anomaly_count_in_window']) if pd.notna(row['anomaly_count_in_window']) else 0,
        'note': note
    })

trace_top10_severity_df = pd.DataFrame(records)

# Keep output compact: selected columns and up to 10 rows
trace_top10_severity_df[['cmdb_id','trace_name','global_threshold','anomaly_timestamps_in_window','earliest_anomaly_timestamp','anomaly_count_in_window','note']].head(10)

```
Out[4]:
```
```python
# Build a concise plain-English summary string from the top-10 severity trace candidates table
# Uses existing kernel variable: trace_top10_severity_df

lines = []
lines.append("Top 10 trace candidates by severity_ratio (non-zero) — window 2021-03-04 22:00:00 to 22:30:00 +08:00:")
lines.append("")

for i, r in trace_top10_severity_df.reset_index(drop=True).iterrows():
    cmdb = r['cmdb_id']
    tname = r['trace_name']
    thr = r['global_threshold']
    count = int(r['anomaly_count_in_window'])
    earliest = r['earliest_anomaly_timestamp']
    # format anomaly timestamps list (limit already applied)
    ts_list = r['anomaly_timestamps_in_window']
    if isinstance(ts_list, list):
        ts_strs = [str(t) for t in ts_list]
    else:
        ts_strs = []
    ts_display = ", ".join(ts_strs) if ts_strs else ""
    note = r.get('note', '')
    lines.append(
        f"{i+1}) {cmdb} | {tname} | threshold={thr:.6g} | anomalies={count} | earliest={earliest} | "
        f"timestamps=[{ts_display}] {('(note: '+note+')') if note else ''}"
    )

lines.append("")
lines.append("Summary notes:")
lines.append("- All top-10 candidates have 2 anomaly points in the window (as shown).")
lines.append("- Timestamps are in UTC+8. If a row shows an empty timestamp list, it means no anomaly timestamps were found for that candidate in the window.")
summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                         trace_name  global_threshold                       anomaly_timestamps_in_window earliest_anomaly_timestamp  anomaly_count_in_window note
0      MG02   trace.from_dockerA1.duration_p95          0.029600  [2021-03-04 22:05:00+08:00, 2021-03-04 22:13:0...  2021-03-04 22:05:00+08:00                        2     
1  dockerA1         trace.to_MG02.duration_p95          0.029600  [2021-03-04 22:05:00+08:00, 2021-03-04 22:13:0...  2021-03-04 22:05:00+08:00                        2     
2      MG02  trace.from_dockerA1.duration_mean          0.013130  [2021-03-04 22:05:00+08:00, 2021-03-04 22:13:0...  2021-03-04 22:05:00+08:00                        2     
3  dockerA1        trace.to_MG02.duration_mean          0.013130  [2021-03-04 22:05:00+08:00, 2021-03-04 22:13:0...  2021-03-04 22:05:00+08:00                        2     
4  dockerA2         trace.to_MG02.duration_p95          0.012755  [2021-03-04 22:08:00+08:00, 2021-03-04 22:15:0...  2021-03-04 22:08:00+08:00                        2     
5      MG02   trace.from_dockerA2.duration_p95          0.012755  [2021-03-04 22:08:00+08:00, 2021-03-04 22:15:0...  2021-03-04 22:08:00+08:00                        2     
6      MG02   trace.from_dockerB1.duration_p95          0.013240  [2021-03-04 22:18:00+08:00, 2021-03-04 22:29:0...  2021-03-04 22:18:00+08:00                        2     
7  dockerB1         trace.to_MG02.duration_p95          0.013240  [2021-03-04 22:18:00+08:00, 2021-03-04 22:29:0...  2021-03-04 22:18:00+08:00                        2     
8      MG02     trace.to_dockerA1.duration_p95          0.435500  [2021-03-04 22:01:00+08:00, 2021-03-04 22:10:0...  2021-03-04 22:01:00+08:00                        2     
9  dockerA1       trace.from_MG02.duration_p95          0.435500  [2021-03-04 22:01:00+08:00, 2021-03-04 22:10:0...  2021-03-04 22:01:00+08:00                        2```
```

In [ ]:
# In[5]:

import pandas as pd

# Reuse kernel variables: df_trace, trace_result, start_ts, end_ts, tz

# Prepare thresholds lookup from trace_result (global_threshold per group)
thr = trace_result[['cmdb_id','trace_name','global_threshold']].drop_duplicates()

# Merge thresholds into full trace dataframe
trace_with_thr = df_trace.merge(thr, on=['cmdb_id','trace_name'], how='left')

# Filter to window
trace_win = trace_with_thr[(trace_with_thr['timestamp'] >= start_ts) & (trace_with_thr['timestamp'] <= end_ts)].copy()

# Mark anomalies: require non-null global_threshold and value >= global_threshold
trace_win['is_anomaly'] = (trace_win['global_threshold'].notna()) & (trace_win['value'] >= trace_win['global_threshold'])

# Aggregate per-minute: total anomalies and distinct cmdb_ids/trace_names (limited to first 10)
def limited_list(series, limit=10):
    uniq = pd.unique(series.dropna())
    return list(uniq[:limit])

agg = trace_win[trace_win['is_anomaly']].groupby('timestamp').agg(
    total_anomalies = ('is_anomaly','sum'),
    cmdb_ids = ('cmdb_id', lambda s: limited_list(s, limit=10)),
    trace_names = ('trace_name', lambda s: limited_list(s, limit=10))
).reset_index()

# Add timezone-aware datetime for display
agg['minute_dt'] = pd.to_datetime(agg['timestamp'], unit='s', utc=True).dt.tz_convert(tz)

# Select top 10 minutes by total_anomalies desc
top_minutes_summary = agg.sort_values(['total_anomalies','timestamp'], ascending=[False, True]).head(10)[
    ['timestamp','minute_dt','total_anomalies','cmdb_ids','trace_names']
].reset_index(drop=True)

# For each of those top minutes, collect anomalous (cmdb_id, trace_name) pairs
top_timestamps = top_minutes_summary['timestamp'].tolist()

pairs = trace_win[trace_win['timestamp'].isin(top_timestamps) & trace_win['is_anomaly']][
    ['timestamp','cmdb_id','trace_name']
].drop_duplicates().sort_values(['timestamp','cmdb_id','trace_name'])

# Limit overall pairs to at most 50 rows (keep earliest timestamps first)
anomalous_pairs_limited = pairs.head(50).reset_index(drop=True)

# Compact outputs: top 10 minutes and up to 50 anomalous pairs
top_minutes_summary, anomalous_pairs_limited

```
Out[5]:
```
```python
# Summarize top-10 anomalous minutes and the anomalous pairs (compact, plain English)
lines = []
lines.append("Top 10 minutes by trace anomaly count (window 2021-03-04 22:00:00 to 22:30:00 +08:00):")
lines.append("")

for i, r in top_minutes_summary.reset_index(drop=True).iterrows():
    minute = r['minute_dt']
    total = int(r['total_anomalies'])
    cmdbs = r['cmdb_ids'] if isinstance(r['cmdb_ids'], list) else []
    traces = r['trace_names'] if isinstance(r['trace_names'], list) else []
    lines.append(
        f"{i+1}) {minute} | total_anomalies={total} | cmdb_count={len(cmdbs)} | cmdb_sample={', '.join(cmdbs[:10])} | "
        f"trace_count={len(traces)} | trace_sample={', '.join(traces[:10])}"
    )

lines.append("")
pairs_count = len(anomalous_pairs_limited)
lines.append(f"Anomalous (cmdb_id, trace_name) pairs for the top minutes (limited to at most 50 rows): total_returned={pairs_count}")
lines.append("Examples (up to first 10 pairs):")
for _, p in anomalous_pairs_limited.head(10).iterrows():
    lines.append(f"- {p['timestamp']} | {p['cmdb_id']} | {p['trace_name']}")

lines.append("")
lines.append("Notes:")
lines.append("- Timestamps are shown in UTC+8.")
lines.append("- cmdb_sample and trace_sample show up to the first 10 distinct items per minute.")
lines.append("- The full per-minute aggregation and up to 50 anomalous pairs are available in the kernel variables "
             "'top_minutes_summary' and 'anomalous_pairs_limited' respectively.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(    timestamp                 minute_dt  total_anomalies                                           cmdb_ids                                        trace_names
0  1614866760 2021-03-04 22:06:00+08:00               70  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...  [trace.self.duration_mean, trace.self.row_coun...
1  1614866880 2021-03-04 22:08:00+08:00               49  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...  [trace.self.duration_mean, trace.self.duration...
2  1614866820 2021-03-04 22:07:00+08:00               42  [IG01, MG01, MG02, Tomcat01, Tomcat02, Tomcat0...  [trace.to_Tomcat01.duration_mean, trace.to_Tom...
3  1614866400 2021-03-04 22:00:00+08:00               36  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat02, T...  [trace.to_Tomcat04.row_count, trace.self.row_c...
4  1614867120 2021-03-04 22:12:00+08:00               31  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat03, d...  [trace.to_Tomcat01.row_count, trace.to_Tomcat0...
5  1614866640 2021-03-04 22:04:00+08:00               28  [IG01, IG02, MG01, MG02, Tomcat02, Tomcat03, T...  [trace.self.row_count, trace.to_Tomcat02.row_c...
6  1614866580 2021-03-04 22:03:00+08:00               24  [IG01, MG01, MG02, Tomcat01, Tomcat03, dockerA...  [trace.to_Tomcat01.row_count, trace.to_Tomcat0...
7  1614867000 2021-03-04 22:10:00+08:00               24  [IG02, MG01, MG02, Tomcat04, dockerA1, dockerA...  [trace.self.duration_p95, trace.to_Tomcat04.du...
8  1614866460 2021-03-04 22:01:00+08:00               22  [IG01, MG01, MG02, Tomcat02, dockerA1, dockerA...  [trace.to_Tomcat02.row_count, trace.to_dockerA...
9  1614866700 2021-03-04 22:05:00+08:00               20  [IG01, IG02, MG01, MG02, Tomcat01, Tomcat04, d...  [trace.to_Tomcat04.row_count, trace.from_docke...,      timestamp   cmdb_id                     trace_name
0   1614866400      IG01    trace.to_Tomcat04.row_count
1   1614866400      IG02           trace.self.row_count
2   1614866400      IG02    trace.to_Tomcat01.row_count
3   1614866400      IG02    trace.to_Tomcat03.row_count
4   1614866400      MG01  trace.from_Tomcat02.row_count
..         ...       ...                            ...
45  1614866460  Tomcat02      trace.from_IG01.row_count
46  1614866460  dockerA1  trace.from_MG02.duration_mean
47  1614866460  dockerA1   trace.from_MG02.duration_p95
48  1614866460  dockerA2  trace.from_MG01.duration_mean
49  1614866460  dockerA2   trace.from_MG01.duration_p95

[50 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import re
import numpy as np
import pandas as pd

# Reuse kernel variables: df_trace, trace_result, tz
# Prepare trace_with_thr (merge global thresholds)
thr = trace_result[['cmdb_id','trace_name','global_threshold']].drop_duplicates()
trace_with_thr = df_trace.merge(thr, on=['cmdb_id','trace_name'], how='left')

# Candidate timestamps
candidates = [1614866460, 1614866760]

def parse_direction_target(trace_name):
    m = re.match(r'^trace\.(to|from)_([A-Za-z0-9]+)', str(trace_name))
    if m:
        direction = m.group(1)  # 'to' or 'from'
        target = m.group(2)
    else:
        direction = 'unknown'
        target = 'unknown'
    return direction, target

results = {}

for ts in candidates:
    # rows at exact timestamp
    subset = trace_with_thr[trace_with_thr['timestamp'] == ts].copy()
    # anomaly condition: threshold exists and value >= threshold
    subset['is_anomaly'] = subset['global_threshold'].notna() & (subset['value'] >= subset['global_threshold'])
    anomalous = subset[subset['is_anomaly']].sort_values('cmdb_id').head(50).copy()
    # add minute dt for readability
    minute_dt = pd.to_datetime(ts, unit='s', utc=True).tz_convert(tz)
    anomalous['minute_dt'] = minute_dt

    # parse direction and target
    parsed = anomalous['trace_name'].apply(parse_direction_target)
    anomalous['direction'] = parsed.apply(lambda x: x[0])
    anomalous['target_component'] = parsed.apply(lambda x: x[1])

    # Build per-timestamp target summary
    if not anomalous.empty:
        grp = anomalous.groupby('target_component')
        target_rows = []
        for target, g in grp:
            to_count = int((g['direction'] == 'to').sum())
            from_count = int((g['direction'] == 'from').sum())
            direction_summary = f"to:{to_count}, from:{from_count}"
            callers_sample = list(pd.unique(g['cmdb_id'])[:5])
            count_pairs = int(len(g))  # count of anomalous rows/pairs at this timestamp for this target
            target_rows.append({
                'target_component': target,
                'direction_summary': direction_summary,
                'callers_sample': callers_sample,
                'count_of_anomalous_pairs_involving_target': count_pairs
            })
        target_summary = pd.DataFrame(target_rows).sort_values(
            'count_of_anomalous_pairs_involving_target', ascending=False
        ).reset_index(drop=True).head(20)
    else:
        target_summary = pd.DataFrame(columns=[
            'target_component','direction_summary','callers_sample','count_of_anomalous_pairs_involving_target'
        ])

    # Top 5 anomalous pairs by severity at this timestamp
    if not anomalous.empty:
        # severity handling: null if global_threshold == 0
        anomalous['global_threshold'] = pd.to_numeric(anomalous['global_threshold'], errors='coerce')
        anomalous['value'] = pd.to_numeric(anomalous['value'], errors='coerce')
        anomalous['severity'] = np.where(
            anomalous['global_threshold'] == 0,
            np.nan,
            (anomalous['value'] - anomalous['global_threshold']) / anomalous['global_threshold']
        )
        top5_severity = anomalous.sort_values(['severity'], ascending=False, na_position='last').head(5)[
            ['cmdb_id','trace_name','value','global_threshold','severity']
        ].reset_index(drop=True)
    else:
        top5_severity = pd.DataFrame(columns=['cmdb_id','trace_name','value','global_threshold','severity'])

    # keep only requested anomalous rows (limit 50) and selected columns for compactness
    anomalous_display = anomalous[
        ['timestamp','minute_dt','cmdb_id','trace_name','value','global_threshold','direction','target_component']
    ].head(50).reset_index(drop=True)

    # store results
    results[ts] = {
        'anomalous_rows': anomalous_display,
        'target_summary': target_summary,
        'top5_severity': top5_severity
    }

# Expose variables for the two timestamps (compact)
anomalous_1614866460 = results[1614866460]['anomalous_rows']
target_summary_1614866460 = results[1614866460]['target_summary']
top5_severity_1614866460 = results[1614866460]['top5_severity']

anomalous_1614866760 = results[1614866760]['anomalous_rows']
target_summary_1614866760 = results[1614866760]['target_summary']
top5_severity_1614866760 = results[1614866760]['top5_severity']

# Return compact tables (variables separated by commas as per IPython convention)
anomalous_1614866460, target_summary_1614866460, top5_severity_1614866460, \
anomalous_1614866760, target_summary_1614866760, top5_severity_1614866760

```
Out[6]:
```
```python
# Summarize findings for timestamps 1614866460 and 1614866760 using existing kernel variables.
# Uses: anomalous_1614866460, target_summary_1614866460, top5_severity_1614866460,
#       anomalous_1614866760, target_summary_1614866760, top5_severity_1614866760

def format_top_targets(df, n=5):
    if df.empty:
        return "none"
    lines = []
    for i, r in df.head(n).iterrows():
        tgt = r['target_component']
        cnt = int(r['count_of_anomalous_pairs_involving_target'])
        callers = r['callers_sample']
        callers_sample = ", ".join(map(str, callers)) if isinstance(callers, (list, tuple)) else str(callers)
        lines.append(f"{tgt} (count={cnt}, callers_sample=[{callers_sample}])")
    return "; ".join(lines)

def format_top_severity(df):
    if df.empty:
        return "none"
    lines = []
    for i, r in df.iterrows():
        sev = r['severity']
        sev_str = f"{sev:.6g}" if pd.notna(sev) else "null"
        lines.append(f"{r['cmdb_id']} | {r['trace_name']} | value={r['value']:.6g} | thr={r['global_threshold']:.6g} | severity={sev_str}")
    return "\n  " + "\n  ".join(lines)

import pandas as pd

t1 = 1614866460
t2 = 1614866760
dt1 = pd.to_datetime(t1, unit='s', utc=True).tz_convert(tz)
dt2 = pd.to_datetime(t2, unit='s', utc=True).tz_convert(tz)

# Stats and small examples
rows_t1 = len(anomalous_1614866460)
rows_t2 = len(anomalous_1614866760)

# Top target summaries (formatted)
top_targets_t1 = format_top_targets(target_summary_1614866460, n=10)
top_targets_t2 = format_top_targets(target_summary_1614866760, n=10)

# Top severity lists
top5_sev_t1 = format_top_severity(top5_severity_1614866460)
top5_sev_t2 = format_top_severity(top5_severity_1614866760)

# Example anomalous row samples (up to 5) for each timestamp
ex1 = anomalous_1614866460.head(5)[['cmdb_id','trace_name','value','global_threshold','direction','target_component']].to_dict('records')
ex2 = anomalous_1614866760.head(5)[['cmdb_id','trace_name','value','global_threshold','direction','target_component']].to_dict('records')

summary_lines = [
    f"Analysis for two candidate minutes (timestamps in UTC+8):",
    f"",
    f"1) {dt1} (ts={t1}):",
    f"   - Total anomalous rows at this minute: {rows_t1} (returned up to 50 rows).",
    f"   - Top target components (by anomalous pair count): {top_targets_t1}",
    f"   - Parsing note: entries with trace.self.* or non 'to_'/'from_' patterns are marked as 'unknown' target/direction.",
    f"   - Example anomalous rows (up to 5):",
]
for r in ex1:
    summary_lines.append(f"     - {r['cmdb_id']} | {r['trace_name']} | value={r['value']:.6g} | thr={r['global_threshold']:.6g} | dir={r['direction']} | target={r['target_component']}")
summary_lines.append("   - Top 5 anomalous pairs by severity (severity = (value - thr)/thr):")
summary_lines.append(top5_sev_t1)
summary_lines.append("")
summary_lines.append(f"2) {dt2} (ts={t2}):")
summary_lines.append(f"   - Total anomalous rows at this minute: {rows_t2} (returned up to 50 rows).")
summary_lines.append(f"   - Top target components (by anomalous pair count): {top_targets_t2}")
summary_lines.append(f"   - Parsing note: entries with trace.self.* or non 'to_'/'from_' patterns are marked as 'unknown' target/direction.")
summary_lines.append(f"   - Example anomalous rows (up to 5):")
for r in ex2:
    summary_lines.append(f"     - {r['cmdb_id']} | {r['trace_name']} | value={r['value']:.6g} | thr={r['global_threshold']:.6g} | dir={r['direction']} | target={r['target_component']}")
summary_lines.append("   - Top 5 anomalous pairs by severity (severity = (value - thr)/thr):")
summary_lines.append(top5_sev_t2)
summary_lines.append("")
summary_lines.append("Concise conclusions:")
summary_lines.append("- At 22:01 (+08:00) there are ~22 anomalous trace rows; MG02 is the top target (mostly 'from' entries),")
summary_lines.append("  and several 'to' relations to dockerA2/dockerB2 appear. Some traces lacked 'to/from' pattern and were marked 'unknown'.")
summary_lines.append("- At 22:06 (+08:00) there are ~50 anomalous trace rows; many involve Tomcat01/Tomcat03 targets and several 'to'/'from' relations between IG and Tomcat services.")
summary_lines.append("- Top severity pairs show clear spikes (e.g., trace.to_MG02.duration_p95 and corresponding from_* entries) indicating unusually high durations or row_counts relative to their global thresholds.")
summary = "\n".join(summary_lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     timestamp                 minute_dt   cmdb_id                       trace_name       value  global_threshold direction target_component
0   1614866460 2021-03-04 22:01:00+08:00      IG01      trace.to_Tomcat02.row_count   40.000000         39.550000        to         Tomcat02
1   1614866460 2021-03-04 22:01:00+08:00      MG01  trace.to_dockerA2.duration_mean    0.099471          0.081731        to         dockerA2
2   1614866460 2021-03-04 22:01:00+08:00      MG01   trace.to_dockerA2.duration_p95    0.466600          0.364940        to         dockerA2
3   1614866460 2021-03-04 22:01:00+08:00      MG01  trace.to_dockerB2.duration_mean    0.205467          0.150529        to         dockerB2
4   1614866460 2021-03-04 22:01:00+08:00      MG02  trace.to_dockerA1.duration_mean    0.124238          0.118269        to         dockerA1
..         ...                       ...       ...                              ...         ...               ...       ...              ...
17  1614866460 2021-03-04 22:01:00+08:00  dockerB1          trace.self.duration_p95    0.014000          0.014000   unknown          unknown
18  1614866460 2021-03-04 22:01:00+08:00  dockerB2    trace.from_MG01.duration_mean    0.205467          0.150529      from             MG01
19  1614866460 2021-03-04 22:01:00+08:00  dockerB2        trace.from_MG02.row_count   51.000000         48.800000      from             MG02
20  1614866460 2021-03-04 22:01:00+08:00  dockerB2          trace.self.duration_p95    0.012000          0.012000   unknown          unknown
21  1614866460 2021-03-04 22:01:00+08:00  dockerB2             trace.self.row_count  801.000000        596.250000   unknown          unknown

[22 rows x 8 columns],   target_component direction_summary                            callers_sample  count_of_anomalous_pairs_involving_target
0             MG02      to:0, from:5  [dockerA1, dockerA2, dockerB1, dockerB2]                                          5
1          unknown      to:0, from:0            [dockerA2, dockerB1, dockerB2]                                          4
2         dockerA2      to:3, from:0                              [MG01, MG02]                                          3
3             MG01      to:0, from:3                      [dockerA2, dockerB2]                                          3
4         dockerB2      to:2, from:0                              [MG01, MG02]                                          2
5         dockerA1      to:2, from:0                                    [MG02]                                          2
6             IG01      to:0, from:1                                [Tomcat02]                                          1
7         Tomcat02      to:1, from:0                                    [IG01]                                          1
8         dockerB1      to:1, from:0                                    [MG02]                                          1,     cmdb_id                       trace_name       value  global_threshold  severity
0  dockerB1        trace.from_MG02.row_count   60.000000         39.400000  0.522843
1      MG02      trace.to_dockerB1.row_count   60.000000         39.400000  0.522843
2      MG01  trace.to_dockerB2.duration_mean    0.205467          0.150529  0.364968
3  dockerB2    trace.from_MG01.duration_mean    0.205467          0.150529  0.364968
4  dockerB2             trace.self.row_count  801.000000        596.250000  0.343396,      timestamp                 minute_dt   cmdb_id                       trace_name       value  global_threshold direction target_component
0   1614866760 2021-03-04 22:06:00+08:00      IG01         trace.self.duration_mean    0.177312          0.150152   unknown          unknown
1   1614866760 2021-03-04 22:06:00+08:00      IG01             trace.self.row_count  324.000000        315.000000   unknown          unknown
2   1614866760 2021-03-04 22:06:00+08:00      IG01  trace.to_Tomcat01.duration_mean    0.173600          0.139111        to         Tomcat01
3   1614866760 2021-03-04 22:06:00+08:00      IG01   trace.to_Tomcat01.duration_p95    0.762750          0.651585        to         Tomcat01
4   1614866760 2021-03-04 22:06:00+08:00      IG01  trace.to_Tomcat03.duration_mean    0.159980          0.150360        to         Tomcat03
..         ...                       ...       ...                              ...         ...               ...       ...              ...
45  1614866760 2021-03-04 22:06:00+08:00  Tomcat02    trace.from_IG02.duration_mean    0.203821          0.164805      from             IG02
46  1614866760 2021-03-04 22:06:00+08:00  Tomcat02     trace.from_IG02.duration_p95    1.137100          0.985495      from             IG02
47  1614866760 2021-03-04 22:06:00+08:00  Tomcat02       trace.to_MG01.duration_p95    0.898650          0.886837        to             MG01
48  1614866760 2021-03-04 22:06:00+08:00  Tomcat02          trace.to_MG02.row_count   51.000000         48.100000        to             MG02
49  1614866760 2021-03-04 22:06:00+08:00  Tomcat03             trace.self.row_count  604.000000        581.650000   unknown          unknown

[50 rows x 8 columns],    target_component direction_summary                          callers_sample  count_of_anomalous_pairs_involving_target
0           unknown      to:0, from:0  [IG01, IG02, MG02, Tomcat01, Tomcat03]                                         10
1          Tomcat01      to:4, from:4                [IG01, IG02, MG01, MG02]                                          8
2          Tomcat03      to:4, from:3                      [IG01, IG02, MG02]                                          7
3          Tomcat04      to:2, from:2                            [IG02, MG02]                                          4
4          Tomcat02      to:2, from:2                      [IG02, MG01, MG02]                                          4
5              IG02      to:0, from:4                    [Tomcat01, Tomcat02]                                          4
6              MG01      to:3, from:0                    [Tomcat01, Tomcat02]                                          3
7          dockerB2      to:1, from:2                            [MG01, MG02]                                          3
8              MG02      to:3, from:0                    [Tomcat01, Tomcat02]                                          3
9              IG01      to:0, from:2                              [Tomcat01]                                          2
10         dockerB1      to:0, from:2                            [MG01, MG02]                                          2,     cmdb_id                         trace_name     value  global_threshold  severity
0  Tomcat01         trace.to_MG02.duration_p95  0.943000          0.389955  1.418228
1      MG02   trace.from_Tomcat01.duration_p95  0.943000          0.389955  1.418228
2      MG02  trace.from_Tomcat01.duration_mean  0.154829          0.089757  0.724985
3  Tomcat01        trace.to_MG02.duration_mean  0.154829          0.089757  0.724985
4  Tomcat01           trace.self.duration_mean  0.023657          0.014320  0.652037)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np

# Reuse variables from kernel: df_metric, start_ts, end_ts, tz
components = ["Tomcat01","Tomcat03","IG01","IG02","MG01","MG02","dockerA1","dockerA2","dockerB1","dockerB2"]

# 1) Compute global P95 thresholds per (cmdb_id, kpi_name) from the full metric series (rule 9)
thr_p95 = df_metric.dropna(subset=['value']).groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index()
thr_p95 = thr_p95.rename(columns={'value':'global_P95'})

# Merge thresholds into full metric dataframe
metric_with_thr = df_metric.merge(thr_p95, on=['cmdb_id','kpi_name'], how='right')  # keep all pairs that have thresholds

# Filter to the incident window and selected components
metric_win = metric_with_thr[
    (metric_with_thr['timestamp'] >= start_ts) &
    (metric_with_thr['timestamp'] <= end_ts) &
    (metric_with_thr['cmdb_id'].isin(components))
].copy()

# Helper to compute per-group aggregates including longest consecutive run
def compute_group_aggregates(all_thresholds_df, window_df):
    records = []
    # iterate over all (cmdb_id,kpi_name) pairs for the selected components that have thresholds
    pairs = all_thresholds_df[['cmdb_id','kpi_name','global_P95']].drop_duplicates()
    for _, pr in pairs.iterrows():
        cmdb = pr['cmdb_id']
        kpi = pr['kpi_name']
        thr = float(pr['global_P95']) if pd.notna(pr['global_P95']) else np.nan
        grp_rows = window_df[(window_df['cmdb_id']==cmdb) & (window_df['kpi_name']==kpi)].copy()
        grp_rows = grp_rows.sort_values('timestamp')
        # anomaly mask in window
        if not grp_rows.empty:
            anomaly_mask = grp_rows['value'] >= thr
            anomaly_count = int(anomaly_mask.sum())
            earliest_anom_ts = int(grp_rows.loc[anomaly_mask, 'timestamp'].min()) if anomaly_mask.any() else pd.NA
            max_val = float(grp_rows['value'].max())
        else:
            anomaly_count = 0
            earliest_anom_ts = pd.NA
            max_val = np.nan

        # severity_ratio: null if thr == 0 or thr is NaN
        if thr == 0 or np.isnan(thr):
            severity = np.nan
        else:
            # If no data in window, severity computed from max_val which may be nan -> result will be nan
            severity = (max_val - thr) / thr if not np.isnan(max_val) else np.nan

        # longest consecutive anomaly run within window: consider only rows in window and where value >= thr
        best_run = 0
        best_run_start = None
        cur_run = 0
        cur_start = None
        prev_ts = None
        if not grp_rows.empty:
            for _, row in grp_rows.iterrows():
                ts = int(row['timestamp'])
                is_anom = (row['value'] >= thr)
                if is_anom:
                    if prev_ts is None or (ts - prev_ts == 60):
                        if cur_run == 0:
                            cur_start = ts
                        cur_run += 1
                    else:
                        # gap between minutes -> start new run
                        cur_run = 1
                        cur_start = ts
                else:
                    cur_run = 0
                    cur_start = None
                if cur_run > best_run:
                    best_run = cur_run
                    best_run_start = cur_start
                prev_ts = ts
        # convert earliest timestamps to tz-aware datetimes or NaT
        def to_dt(x):
            if pd.isna(x) or x is pd.NA:
                return pd.NaT
            return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)
        earliest_anom_dt = to_dt(earliest_anom_ts)
        best_run_start_dt = to_dt(best_run_start)

        records.append({
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'global_P95': thr,
            'anomaly_count_in_window': anomaly_count,
            'earliest_anomaly_timestamp_in_window': earliest_anom_dt,
            'max_value_in_window': max_val,
            'severity_ratio': severity,
            'longest_consecutive_anomaly_length': int(best_run),
            'earliest_timestamp_of_longest_run': best_run_start_dt
        })
    return pd.DataFrame.from_records(records)

# Compute aggregates for selected components pairs (use thr_p95 filtered)
thr_selected = thr_p95[thr_p95['cmdb_id'].isin(components)].copy()
agg_df = compute_group_aggregates(thr_selected, metric_win)

# A) Top 20 rows across all listed components sorted by severity_ratio desc then longest_consecutive desc
# Treat NaN severity as lowest (they will be placed last)
agg_sorted = agg_df.sort_values(['severity_ratio','longest_consecutive_anomaly_length'],
                                ascending=[False, False], na_position='last').reset_index(drop=True)
top20_overall = agg_sorted.head(20)

# B) Detailed tables for Tomcat01 and MG02 (limit 50 rows each)
tomcat01_df = agg_df[agg_df['cmdb_id']=='Tomcat01'].sort_values(
    ['severity_ratio','longest_consecutive_anomaly_length'], ascending=[False,False], na_position='last'
).reset_index(drop=True).head(50)

mg02_df = agg_df[agg_df['cmdb_id']=='MG02'].sort_values(
    ['severity_ratio','longest_consecutive_anomaly_length'], ascending=[False,False], na_position='last'
).reset_index(drop=True).head(50)

# Keep outputs compact: selected columns and limited rows as requested
cols = ['cmdb_id','kpi_name','global_P95','anomaly_count_in_window','earliest_anomaly_timestamp_in_window',
        'max_value_in_window','severity_ratio','longest_consecutive_anomaly_length','earliest_timestamp_of_longest_run']

top20_overall = top20_overall[cols]
tomcat01_df = tomcat01_df[cols]
mg02_df = mg02_df[cols]

# Display the three compact DataFrames (IPython will show them)
top20_overall, tomcat01_df, mg02_df

```
Out[7]:
```
```python
# Summarize the metric analysis results in plain English using kernel DataFrames:
# top20_overall, tomcat01_df, mg02_df

def fmt_val(v, fmt="{:.6g}"):
    try:
        if pd.isna(v):
            return "null"
        return fmt.format(v)
    except:
        return str(v)

lines = []
lines.append("Metric analysis summary (window 2021-03-04 22:00:00 to 22:30:00 +08:00):")
lines.append("")

# Top overall issues (from top20_overall)
lines.append("A) Top issues across selected components (top 3 examples):")
if top20_overall.shape[0] == 0:
    lines.append("  - No metric anomalies found.")
else:
    for i, r in top20_overall.head(3).iterrows():
        lines.append(
            f"  {i+1}) {r['cmdb_id']} | {r['kpi_name']} | P95={fmt_val(r['global_P95'])} | "
            f"max={fmt_val(r['max_value_in_window'])} | severity_ratio={fmt_val(r['severity_ratio'])} | "
            f"anomalies={int(r['anomaly_count_in_window'])} | longest_run={int(r['longest_consecutive_anomaly_length'])} "
            f"(start={r['earliest_timestamp_of_longest_run']})"
        )
    lines.append(f"  - (Total rows in top table: {len(top20_overall)})")
lines.append("")

# Tomcat01 detailed
cnt_t1 = len(tomcat01_df)
lines.append(f"B) Tomcat01 detailed (returned {cnt_t1} KPI pairs, limited to 50):")
if cnt_t1 == 0:
    lines.append("  - No KPI pairs for Tomcat01 in the selected components/window.")
else:
    # top 3 by severity
    for i, r in tomcat01_df.head(3).iterrows():
        lines.append(
            f"  {i+1}) {r['kpi_name']} | P95={fmt_val(r['global_P95'])} | max={fmt_val(r['max_value_in_window'])} | "
            f"severity_ratio={fmt_val(r['severity_ratio'])} | anomalies={int(r['anomaly_count_in_window'])} | "
            f"longest_run={int(r['longest_consecutive_anomaly_length'])} (start={r['earliest_timestamp_of_longest_run']})"
        )
    lines.append("  - Note: many Tomcat01 KPIs have non-zero anomaly counts; several show consecutive short runs (length 1–3).")
lines.append("")

# MG02 detailed
cnt_mg = len(mg02_df)
lines.append(f"C) MG02 detailed (returned {cnt_mg} KPI pairs, limited to 50):")
if cnt_mg == 0:
    lines.append("  - No KPI pairs for MG02 in the selected components/window.")
else:
    for i, r in mg02_df.head(3).iterrows():
        lines.append(
            f"  {i+1}) {r['kpi_name']} | P95={fmt_val(r['global_P95'])} | max={fmt_val(r['max_value_in_window'])} | "
            f"severity_ratio={fmt_val(r['severity_ratio'])} | anomalies={int(r['anomaly_count_in_window'])} | "
            f"longest_run={int(r['longest_consecutive_anomaly_length'])} (start={r['earliest_timestamp_of_longest_run']})"
        )
    lines.append("  - Note: MG02 shows both disk and CPU-related high-severity spikes and also long-running JVM memory metrics (31-minute runs) with severity 0 (max == P95).")
lines.append("")

lines.append("Overall conclusions:")
lines.append("- The highest severity metric (by severity_ratio) is an IG02 CPU WIO KPI (severity >5x), followed by several disk I/O spikes on IG02 and MG02.")
lines.append("- Tomcat instances show multiple KPI anomalies (e.g., thread counts, CPU WIO, disk metrics) but generally shorter runs.")
lines.append("- MG02 contains both high-severity short spikes and several metrics (JVM memory) with full-window consecutive anomalies where observed max equals P95 (severity 0).")
lines.append("- All thresholds used are global P95 computed from the full series, and all aggregations are within the specified incident window.")
lines.append("")
summary = "\n".join(lines)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  global_P95  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  severity_ratio  longest_consecutive_anomaly_length earliest_timestamp_of_longest_run
0       IG02                             OSLinux-CPU_CPU_CPUWio    0.091448                        2            2021-03-04 22:00:00+08:00               0.5652        5.180562                                   1         2021-03-04 22:00:00+08:00
1       IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  130.733350                        1            2021-03-04 22:10:00+08:00             504.1333        2.856195                                   1         2021-03-04 22:10:00+08:00
2       IG02    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRTps    0.991700                        1            2021-03-04 22:10:00+08:00               3.3167        2.344459                                   1         2021-03-04 22:10:00+08:00
3       MG02                             OSLinux-CPU_CPU_CPUWio    0.199600                        2            2021-03-04 22:11:00+08:00               0.5655        1.833166                                   1         2021-03-04 22:11:00+08:00
4   Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps    0.400000                        1            2021-03-04 22:11:00+08:00               1.0000        1.500000                                   1         2021-03-04 22:11:00+08:00
5   Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    0.400000                        1            2021-03-04 22:07:00+08:00               1.0000        1.500000                                   1         2021-03-04 22:07:00+08:00
6       IG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  232.000000                        1            2021-03-04 22:11:00+08:00             559.0000        1.409483                                   1         2021-03-04 22:11:00+08:00
7       IG02   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite   24.200025                        1            2021-03-04 22:10:00+08:00              55.2000        1.280989                                   1         2021-03-04 22:10:00+08:00
8       MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps    0.450000                        1            2021-03-04 22:00:00+08:00               1.0000        1.222222                                   1         2021-03-04 22:00:00+08:00
9       MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...    0.050000                        1            2021-03-04 22:28:00+08:00               0.1000        1.000000                                   1         2021-03-04 22:28:00+08:00
10  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...    0.500000                        2            2021-03-04 22:03:00+08:00               1.0000        1.000000                                   1         2021-03-04 22:03:00+08:00
11      MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  101.000000                        1            2021-03-04 22:22:00+08:00             200.0000        0.980198                                   1         2021-03-04 22:22:00+08:00
12      MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  143.500000                        1            2021-03-04 22:17:00+08:00             266.0000        0.853659                                   1         2021-03-04 22:17:00+08:00
13  Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps    0.550000                        1            2021-03-04 22:27:00+08:00               1.0000        0.818182                                   1         2021-03-04 22:27:00+08:00
14      MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps    2.178000                        1            2021-03-04 22:18:00+08:00               3.9600        0.818182                                   1         2021-03-04 22:18:00+08:00
15      MG01    OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRead  139.760000                        1            2021-03-04 22:17:00+08:00             249.2000        0.783057                                   1         2021-03-04 22:17:00+08:00
16      MG01          JVM-Operating System_7779_JVM_JVM_CPULoad    7.551330                        1            2021-03-04 22:04:00+08:00              13.4267        0.778058                                   1         2021-03-04 22:04:00+08:00
17  Tomcat03                    OSLinux-CPU_CPU-2_SingleCpuidle   12.848545                        1            2021-03-04 22:19:00+08:00              22.7764        0.772683                                   1         2021-03-04 22:19:00+08:00
18  Tomcat01                             OSLinux-CPU_CPU_CPUWio    0.033300                        2            2021-03-04 22:01:00+08:00               0.0582        0.747748                                   1         2021-03-04 22:01:00+08:00
19  Tomcat03                    OSLinux-CPU_CPU-1_SingleCpuUtil   13.375675                        1            2021-03-04 22:19:00+08:00              23.3533        0.745953                                   1         2021-03-04 22:19:00+08:00,      cmdb_id                                           kpi_name  global_P95  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  severity_ratio  longest_consecutive_anomaly_length earliest_timestamp_of_longest_run
0   Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     0.50000                        2            2021-03-04 22:03:00+08:00               1.0000        1.000000                                   1         2021-03-04 22:03:00+08:00
1   Tomcat01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps     0.55000                        1            2021-03-04 22:27:00+08:00               1.0000        0.818182                                   1         2021-03-04 22:27:00+08:00
2   Tomcat01                             OSLinux-CPU_CPU_CPUWio     0.03330                        2            2021-03-04 22:01:00+08:00               0.0582        0.747748                                   1         2021-03-04 22:01:00+08:00
3   Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...     0.43000                        1            2021-03-04 22:09:00+08:00               0.7000        0.627907                                   1         2021-03-04 22:09:00+08:00
4   Tomcat01   OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKWrite    39.86325                        1            2021-03-04 22:09:00+08:00              64.1333        0.608833                                   1         2021-03-04 22:09:00+08:00
..       ...                                                ...         ...                      ...                                  ...                  ...             ...                                 ...                               ...
45  Tomcat01          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem    58.00000                       22            2021-03-04 22:00:00+08:00              58.0000        0.000000                                   3         2021-03-04 22:12:00+08:00
46  Tomcat01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     0.30000                       21            2021-03-04 22:00:00+08:00               0.3000        0.000000                                   3         2021-03-04 22:27:00+08:00
47  Tomcat01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     3.00000                       22            2021-03-04 22:00:00+08:00               3.0000        0.000000                                   3         2021-03-04 22:12:00+08:00
48  Tomcat01                            OSLinux-CPU_CPU_CPULoad     1.06000                        3            2021-03-04 22:00:00+08:00               1.0600        0.000000                                   2         2021-03-04 22:00:00+08:00
49  Tomcat01       OSLinux-OSLinux_MEMORY_MEMORY_NoCacheMemPerc    42.08500                        6            2021-03-04 22:22:00+08:00              42.0850        0.000000                                   2         2021-03-04 22:24:00+08:00

[50 rows x 9 columns],    cmdb_id                                           kpi_name    global_P95  anomaly_count_in_window earliest_anomaly_timestamp_in_window  max_value_in_window  severity_ratio  longest_consecutive_anomaly_length earliest_timestamp_of_longest_run
0     MG02                             OSLinux-CPU_CPU_CPUWio  1.996000e-01                        2            2021-03-04 22:11:00+08:00         5.655000e-01        1.833166                                   1         2021-03-04 22:11:00+08:00
1     MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKTps  4.500000e-01                        1            2021-03-04 22:00:00+08:00         1.000000e+00        1.222222                                   1         2021-03-04 22:00:00+08:00
2     MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKPer...  5.000000e-02                        1            2021-03-04 22:28:00+08:00         1.000000e-01        1.000000                                   1         2021-03-04 22:28:00+08:00
3     MG02  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  1.010000e+02                        1            2021-03-04 22:22:00+08:00         2.000000e+02        0.980198                                   1         2021-03-04 22:22:00+08:00
4     MG02     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps  2.178000e+00                        1            2021-03-04 22:18:00+08:00         3.960000e+00        0.818182                                   1         2021-03-04 22:18:00+08:00
..     ...                                                ...           ...                      ...                                  ...                  ...             ...                                 ...                               ...
45    MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed  9.468300e+08                       31            2021-03-04 22:00:00+08:00         9.468300e+08        0.000000                                  31         2021-03-04 22:00:00+08:00
46    MG02       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01                       22            2021-03-04 22:00:00+08:00         9.800000e+01        0.000000                                   6         2021-03-04 22:18:00+08:00
47    MG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies  1.000000e+00                       21            2021-03-04 22:00:00+08:00         1.000000e+00        0.000000                                   3         2021-03-04 22:11:00+08:00
48    MG02          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem  5.800000e+01                       21            2021-03-04 22:00:00+08:00         5.800000e+01        0.000000                                   3         2021-03-04 22:11:00+08:00
49    MG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount  3.000000e+00                       21            2021-03-04 22:00:00+08:00         3.000000e+00        0.000000                                   3         2021-03-04 22:11:00+08:00

[50 rows x 9 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# Reuse kernel variables where available: df_metric, thr_p95, start_ts, end_ts, tz
# If any are missing, load/define minimally (but per instructions they exist in kernel).
tz = pytz.timezone('Asia/Shanghai')

components = ["IG02","MG02","Tomcat01","Tomcat03","IG01","MG01","dockerA1","dockerA2","dockerB1","dockerB2"]

# Ensure numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Use existing global P95 thresholds (thr_p95) computed from full series
# Filter thresholds to our components
thr_selected = thr_p95[thr_p95['cmdb_id'].isin(components)].copy()

# Prepare windowed metric rows for selected components
metric_win_sel = df_metric[
    (df_metric['timestamp'] >= start_ts) &
    (df_metric['timestamp'] <= end_ts) &
    (df_metric['cmdb_id'].isin(components))
].copy()

# Compute per-(cmdb_id, kpi_name) aggregates within the window using the precomputed global P95
records = []
for _, row in thr_selected.iterrows():
    cmdb = row['cmdb_id']
    kpi = row['kpi_name']
    thr = float(row['global_P95']) if pd.notna(row['global_P95']) else np.nan

    grp = metric_win_sel[(metric_win_sel['cmdb_id']==cmdb) & (metric_win_sel['kpi_name']==kpi)].sort_values('timestamp')
    if not grp.empty:
        # anomaly mask
        anom_mask = grp['value'] >= thr
        anomaly_count = int(anom_mask.sum())
        earliest_anom_ts = int(grp.loc[anom_mask, 'timestamp'].min()) if anom_mask.any() else pd.NA
        max_val = float(grp['value'].max())
    else:
        anomaly_count = 0
        earliest_anom_ts = pd.NA
        max_val = np.nan

    # severity ratio (null if thr == 0 or thr is nan)
    if thr == 0 or np.isnan(thr):
        severity = np.nan
    else:
        severity = (max_val - thr) / thr if not np.isnan(max_val) else np.nan

    # compute longest consecutive anomaly length within window (consecutive timestamps differ by 60s)
    best_run = 0
    best_start = None
    cur_run = 0
    cur_start = None
    prev_ts = None
    if not grp.empty:
        for _, r in grp.iterrows():
            ts = int(r['timestamp'])
            is_anom = (r['value'] >= thr)
            if is_anom:
                if (prev_ts is None) or (ts - prev_ts == 60):
                    if cur_run == 0:
                        cur_start = ts
                    cur_run += 1
                else:
                    cur_run = 1
                    cur_start = ts
            else:
                cur_run = 0
                cur_start = None
            if cur_run > best_run:
                best_run = cur_run
                best_start = cur_start
            prev_ts = ts

    def to_dt(x):
        if pd.isna(x) or x is pd.NA:
            return pd.NaT
        return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz)

    records.append({
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'global_P95': thr,
        'max_value_in_window': max_val,
        'severity_ratio': severity,
        'anomaly_count_in_window': anomaly_count,
        'earliest_anomaly_timestamp_in_window': to_dt(earliest_anom_ts),
        'longest_consecutive_anomaly_length': int(best_run),
        'earliest_timestamp_of_longest_run': to_dt(best_start)
    })

agg_pairs = pd.DataFrame.from_records(records)

# For each component, pick the KPI with maximum severity_ratio (ignore NaN). If all NaN for a component, pick KPI with max_value_in_window.
component_rows = []
for comp in components:
    comp_df = agg_pairs[agg_pairs['cmdb_id']==comp].copy()
    if comp_df.empty:
        continue
    # prefer rows with non-null severity; choose max severity
    if comp_df['severity_ratio'].dropna().any():
        best_idx = comp_df['severity_ratio'].idxmax()
    else:
        # fallback: choose KPI with largest max_value_in_window
        best_idx = comp_df['max_value_in_window'].idxmax()
    best_row = comp_df.loc[best_idx]
    severity = best_row['severity_ratio']
    severe_gt_50pct = bool(severity > 0.5) if (not pd.isna(severity)) else False
    component_rows.append({
        'component': comp,
        'kpi_name': best_row['kpi_name'],
        'global_P95': best_row['global_P95'],
        'max_value_in_window': best_row['max_value_in_window'],
        'severity_ratio': severity,
        'earliest_anomaly_timestamp_in_window': best_row['earliest_anomaly_timestamp_in_window'],
        'longest_consecutive_anomaly_length': best_row['longest_consecutive_anomaly_length'],
        'earliest_timestamp_of_longest_run': best_row['earliest_timestamp_of_longest_run'],
        'severe_gt_50pct': severe_gt_50pct
    })

component_summary = pd.DataFrame.from_records(component_rows)

# Sort by severity_ratio desc (NaN last). Limit to 20 rows
component_summary_sorted = component_summary.sort_values('severity_ratio', ascending=False, na_position='last').reset_index(drop=True).head(20)

# Keep output compact: select and order columns
out_cols = ['component','kpi_name','global_P95','max_value_in_window','severity_ratio',
            'earliest_anomaly_timestamp_in_window','longest_consecutive_anomaly_length',
            'earliest_timestamp_of_longest_run','severe_gt_50pct']
component_summary_sorted = component_summary_sorted[out_cols]

# Display the compact summary (one table)
component_summary_sorted

```
Out[8]:
```
```python
# Build a plain-English summary string from the previously computed component_summary_sorted
import pandas as pd
import math

df = component_summary_sorted  # from kernel

def fmt(x, sig=6):
    if pd.isna(x):
        return "null"
    if isinstance(x, (int,)) and abs(x) > 1e6:
        return f"{x:.0f}"
    try:
        if abs(float(x)) >= 1e-3 and abs(float(x)) < 1e6:
            return f"{float(x):.{sig}g}"
        else:
            return f"{float(x):.6g}"
    except:
        return str(x)

lines = []
lines.append("Per-component top KPI by severity_ratio within window 2021-03-04 22:00:00 to 22:30:00 +08:00:")
lines.append("")

for i, r in df.iterrows():
    comp = r['component']
    kpi = r['kpi_name']
    p95 = fmt(r['global_P95'])
    mx = fmt(r['max_value_in_window'])
    sev = "null" if pd.isna(r['severity_ratio']) else f"{float(r['severity_ratio']):.4g}"
    earliest = r['earliest_anomaly_timestamp_in_window']
    earliest_str = str(earliest) if (not pd.isna(earliest)) else "none"
    longest = int(r['longest_consecutive_anomaly_length'])
    longest_start = r['earliest_timestamp_of_longest_run']
    longest_start_str = str(longest_start) if (not pd.isna(longest_start)) else "none"
    severe_flag = bool(r['severe_gt_50pct'])
    lines.append(
        f"- {comp}: KPI='{kpi}' | P95={p95} | max={mx} | severity_ratio={sev} | "
        f"earliest_anomaly={earliest_str} | longest_run={longest} (start={longest_start_str}) | severe_gt_50pct={severe_flag}"
    )

lines.append("")
lines.append("Top observations:")
lines.append("- IG02 and MG02 show the largest severity ratios (IG02 CPU WIO ~5.18x, MG02 CPU WIO ~1.83x) — both flagged severe_gt_50pct=True.")
lines.append("- Tomcat03 and Tomcat01 also show notable spikes (Tomcat03 disk TPS 1.5x; Tomcat01 thread metric 1.0x).")
lines.append("- Several container metrics (docker*) show small or zero severity; dockerB2's top KPI has severity 0 (max == P95).")
lines.append("- Timestamps are in UTC+8 and all thresholds are global P95 computed from full series before window filtering.")

summary = "\n".join(lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

component                                           kpi_name    global_P95  max_value_in_window  severity_ratio earliest_anomaly_timestamp_in_window  longest_consecutive_anomaly_length earliest_timestamp_of_longest_run  severe_gt_50pct
0      IG02                             OSLinux-CPU_CPU_CPUWio  9.144800e-02         5.652000e-01        5.180562            2021-03-04 22:00:00+08:00                                   1         2021-03-04 22:00:00+08:00             True
1      MG02                             OSLinux-CPU_CPU_CPUWio  1.996000e-01         5.655000e-01        1.833166            2021-03-04 22:11:00+08:00                                   1         2021-03-04 22:11:00+08:00             True
2  Tomcat03     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKTps  4.000000e-01         1.000000e+00        1.500000            2021-03-04 22:11:00+08:00                                   1         2021-03-04 22:11:00+08:00             True
3  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  5.000000e-01         1.000000e+00        1.000000            2021-03-04 22:03:00+08:00                                   1         2021-03-04 22:03:00+08:00             True
4      MG01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKRea...  1.435000e+02         2.660000e+02        0.853659            2021-03-04 22:17:00+08:00                                   1         2021-03-04 22:17:00+08:00             True
5      IG01     OSLinux-OSLinux_LOCALDISK_LOCALDISK-sda_DSKBps  3.050000e+00         5.000000e+00        0.639344            2021-03-04 22:13:00+08:00                                   1         2021-03-04 22:13:00+08:00             True
6  dockerB1  Container-DOCKER_CONTAINER_b30097144a13--bcou-...  5.197640e+04         5.774200e+04        0.110927            2021-03-04 22:00:00+08:00                                   1         2021-03-04 22:00:00+08:00            False
7  dockerA1  Container-DOCKER_CONTAINER_7b4b80f345e0--bcou-...  4.580000e-01         5.000000e-01        0.091703            2021-03-04 22:21:00+08:00                                   1         2021-03-04 22:21:00+08:00            False
8  dockerA2  Container-DOCKER_CONTAINER_cb2bbb5e3f90--bcou-...  2.791000e+00         2.870000e+00        0.028305            2021-03-04 22:24:00+08:00                                   1         2021-03-04 22:24:00+08:00            False
9  dockerB2  Container-DOCKER_CONTAINER_350771a68ac2--bcou-...  3.072000e+09         3.072000e+09        0.000000            2021-03-04 22:16:00+08:00                                   1         2021-03-04 22:16:00+08:00            False```
```

In [ ]:
# In[9]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# Reuse existing kernel variables: df_metric, thr_p95, start_ts, end_ts
# Merge thresholds if not already merged
metric_with_thr = df_metric.merge(thr_p95, on=['cmdb_id','kpi_name'], how='left')

# Candidate timestamps
candidate_ts = [1614866460, 1614866760]

# Containers for outputs
anomalies_outputs = {}
summary_records = []

for ts in candidate_ts:
    # Filter rows exactly at this timestamp with a known global_P95
    rows_at_ts = metric_with_thr[metric_with_thr['timestamp'] == ts].copy()
    # Keep only rows where threshold exists and value >= global_P95
    cond = rows_at_ts['global_P95'].notna() & (rows_at_ts['value'] >= rows_at_ts['global_P95'])
    anomalies = rows_at_ts[cond].copy()
    if not anomalies.empty:
        # compute severity_ratio, null if global_P95 == 0
        anomalies['global_P95'] = pd.to_numeric(anomalies['global_P95'], errors='coerce')
        anomalies['value'] = pd.to_numeric(anomalies['value'], errors='coerce')
        anomalies['severity_ratio'] = np.where(
            anomalies['global_P95'] == 0,
            np.nan,
            (anomalies['value'] - anomalies['global_P95']) / anomalies['global_P95']
        )
        # Order by severity desc (NaNs last), limit 100
        anomalies = anomalies.sort_values('severity_ratio', ascending=False, na_position='last').head(100)
        # select requested columns
        anomalies_display = anomalies[['timestamp','cmdb_id','kpi_name','value','global_P95','severity_ratio']].reset_index(drop=True)
        # store
        anomalies_outputs[ts] = anomalies_display
        # Aggregate summary for this timestamp
        distinct_components = list(pd.unique(anomalies_display['cmdb_id']))
        num_distinct = len(distinct_components)
        comp_list_limited = distinct_components[:20]
        # top component by severity (first row)
        top_row = anomalies_display.iloc[0]
        top_comp = top_row['cmdb_id']
        top_kpi = top_row['kpi_name']
        top_sev = top_row['severity_ratio']
        # earliest anomaly timestamp for that (component,kpi) within the incident window (scan window)
        mask_window = (
            (metric_with_thr['cmdb_id'] == top_comp) &
            (metric_with_thr['kpi_name'] == top_kpi) &
            (metric_with_thr['timestamp'] >= start_ts) &
            (metric_with_thr['timestamp'] <= end_ts) &
            (metric_with_thr['global_P95'].notna()) &
            (metric_with_thr['value'] >= metric_with_thr['global_P95'])
        )
        earliest_ts_row = metric_with_thr.loc[mask_window, 'timestamp']
        if not earliest_ts_row.empty:
            earliest_ts_val = int(earliest_ts_row.min())
            earliest_dt = pd.to_datetime(earliest_ts_val, unit='s', utc=True).tz_convert(tz)
        else:
            earliest_dt = pd.NaT
        summary_records.append({
            'timestamp': ts,
            'minute_dt': pd.to_datetime(ts, unit='s', utc=True).tz_convert(tz),
            'number_of_distinct_components_with_metric_anomalies': num_distinct,
            'list_of_distinct_components': comp_list_limited,
            'top_component': top_comp,
            'top_kpi_name': top_kpi,
            'top_severity_ratio': top_sev,
            'earliest_anomaly_timestamp_for_top_component_within_window': earliest_dt,
            'note': ''
        })
    else:
        # No anomalies at this timestamp
        anomalies_outputs[ts] = anomalies[['timestamp','cmdb_id','kpi_name','value','global_P95']].head(0)
        summary_records.append({
            'timestamp': ts,
            'minute_dt': pd.to_datetime(ts, unit='s', utc=True).tz_convert(tz),
            'number_of_distinct_components_with_metric_anomalies': 0,
            'list_of_distinct_components': [],
            'top_component': pd.NA,
            'top_kpi_name': pd.NA,
            'top_severity_ratio': pd.NA,
            'earliest_anomaly_timestamp_for_top_component_within_window': pd.NaT,
            'note': 'No metric KPI breached P95 at this minute'
        })

# Build summary DataFrame (compact)
summary_df = pd.DataFrame(summary_records)[
    ['timestamp','minute_dt','number_of_distinct_components_with_metric_anomalies',
     'list_of_distinct_components','top_component','top_kpi_name','top_severity_ratio',
     'earliest_anomaly_timestamp_for_top_component_within_window','note']
]

# Expose compact outputs: anomalies for each timestamp (limited rows) and the summary table
anomalies_1614866460 = anomalies_outputs[1614866460]
anomalies_1614866760 = anomalies_outputs[1614866760]
summary_df

```
Out[9]:
```
```python
# Concise plain-English summary of metric anomalies at the two candidate minutes
summary = (
    "Minute 2021-03-04 22:01:00 (+08:00, ts=1614866460):\n"
    "- Metric anomalies detected across 15 distinct components (returned up to 100 rows).\n"
    "- Top component by severity: Mysql01, KPI=OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps, "
    "severity_ratio=2.333333 (earliest anomaly at 2021-03-04 22:01:00+08:00).\n\n"
    "Minute 2021-03-04 22:06:00 (+08:00, ts=1614866760):\n"
    "- Metric anomalies detected across 22 distinct components (returned up to 100 rows).\n"
    "- Top component by severity: Mysql01, KPI=Mysql-MySQL_3306_Handler Read Rnd, "
    "severity_ratio≈4.959813 (earliest anomaly at 2021-03-04 22:06:00+08:00).\n\n"
    "Notes:\n"
    "- Detailed anomalous metric rows for each minute are available in the kernel variables "
    "'anomalies_1614866460' and 'anomalies_1614866760' (each limited to 100 rows and ordered by severity).\n"
    "- All thresholds used are global P95 values computed from the full series before window filtering."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                 minute_dt  number_of_distinct_components_with_metric_anomalies                        list_of_distinct_components top_component                                    top_kpi_name  top_severity_ratio earliest_anomaly_timestamp_for_top_component_within_window note
0  1614866460 2021-03-04 22:01:00+08:00                                                 15    [Mysql01, Redis01, Tomcat01, apache01, Redis02...       Mysql01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKBps            2.333333                          2021-03-04 22:01:00+08:00             
1  1614866760 2021-03-04 22:06:00+08:00                                                 22    [Mysql01, ServiceTest2, Mysql02, IG02, Redis01...       Mysql01               Mysql-MySQL_3306_Handler Read Rnd            4.959813                          2021-03-04 22:06:00+08:00```
```